In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
sys.path.append("..") 

from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel, FinalAnswerTool, GoogleSearchTool, VisitWebpageTool
import datetime
import requests
import pytz
import random
import yaml
import PIL
import numpy as np
from collections.abc import Iterable

import torch

from agents.utils import export_masks

from agents.tools.refiner.kitti_tracking import KittiDataset

from agents.Gradio_UI import GradioUI

from matplotlib import pyplot as plt

In [ ]:
root_dir = "E:/KittiTracking"
n_steps, n_pred_steps = 16, 3

train_ds = KittiDataset(root_dir, "train", n_steps, n_pred_steps)
for sample in train_ds:
	frames, _ = sample
	break

In [ ]:
model = HfApiModel(
	max_tokens=4906,
	temperature=0.5,
	model_id="meta-llama/Meta-Llama-3-8B-Instruct", # it is possible that this model may be overloaded
	custom_role_conversions=None,
)

with open("./interpreter.yaml", 'r') as stream:
	prompt_templates = yaml.safe_load(stream)

final_answer = FinalAnswerTool()

# final_answer 
agent = CodeAgent(
	model=model,
	tools=[
		final_answer,
	],
	max_steps=6,
	verbosity_level=2,
    grammar=None,
	planning_interval=None,
	name=None,
	description=None,
	prompt_templates=prompt_templates,
)

result = agent.run(
	"Indoor Service Robot"
)